# AutoML That Tells You Why

**Most AutoML picks the highest number. FerroML picks the best model that's actually distinguishable.**

---

When you run model selection -- whether through AutoML or a manual grid search -- you get a leaderboard. Model A scored 0.73, Model B scored 0.67. So Model A wins, right?

Not so fast. Those scores come from cross-validation, which means they're *estimates* with *uncertainty*. If you ran the experiment again with different random splits, Model B might come out on top. The question isn't "which number is bigger?" -- it's **"are these numbers actually different?"**

This notebook demonstrates how FerroML approaches model selection with statistical rigor:

1. **Run AutoML** to search over multiple algorithm families
2. **Quantify uncertainty** with confidence intervals on every score
3. **Correct for multiple comparisons** -- when you compare 9 models pairwise, you're running 36 tests. Some will be "significant" by chance alone.
4. **Identify the competitive set** -- models that are *not statistically distinguishable* from the best

The result: instead of false confidence in a single winner, you get an honest answer about which models are actually in the running.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from itertools import combinations

from ferroml.automl import AutoML, AutoMLConfig

## Step 1: A Dataset Where Model Choice Actually Matters

We generate a regression target with nonlinear components (sin, quadratic, interaction terms) plus noise. This creates a realistic scenario where:
- **Linear models** capture only part of the signal
- **Tree-based models** have an edge, but not a landslide
- **The gap between the top models is small** -- exactly the regime where naive "pick the max" fails

In [ ]:
np.random.seed(42)
n_samples, n_features = 400, 8
X = np.random.randn(n_samples, n_features)

y = (
    3.0 * np.sin(X[:, 0])          # nonlinear
    + 2.0 * X[:, 1] ** 2           # quadratic
    + 1.5 * X[:, 2] * X[:, 3]      # interaction
    + 0.8 * X[:, 4]                 # linear
    + np.random.randn(n_samples)    # noise
)

print(f"Dataset: {X.shape[0]} samples x {X.shape[1]} features")
print(f"Target range: [{y.min():.1f}, {y.max():.1f}]")

## Step 2: Run FerroML AutoML

FerroML's AutoML searches over algorithm families, tunes hyperparameters, and -- critically -- runs statistical significance tests on every model comparison. The `multiple_testing="bh"` flag enables **Benjamini-Hochberg** false discovery rate control.

In [ ]:
config = AutoMLConfig(
    task="regression",
    metric="r2",
    time_budget_seconds=30,       # 30 seconds is enough to explore the portfolio
    cv_folds=10,                  # 10-fold CV for tighter confidence intervals
    statistical_tests=True,       # enable pairwise significance tests
    confidence_level=0.95,        # 95% confidence intervals
    multiple_testing="bh",        # Benjamini-Hochberg FDR correction
    seed=42,
)

automl = AutoML(config)
result = automl.fit(X, y)
print(result.summary())

### AutoML found the best *configuration* per algorithm -- but how confident are we?

Let's extract the best trial for each algorithm family and look at the spread:

In [ ]:
# Best trial per algorithm family
algo_best = {}
for entry in result.leaderboard:
    algo = entry.algorithm
    if algo not in algo_best or entry.cv_score > algo_best[algo].cv_score:
        algo_best[algo] = entry

ranking = sorted(algo_best.values(), key=lambda e: -e.cv_score)

print(f"{'#':<4} {'Algorithm':<35} {'R² (mean +/- std)':<22} {'95% CI'}")
print("-" * 82)
for i, e in enumerate(ranking, 1):
    print(f"{i:<4} {e.algorithm:<35} {e.cv_score:.4f} +/- {e.cv_std:.4f}    [{e.ci_lower:.4f}, {e.ci_upper:.4f}]")

## Step 3: Deeper Analysis with Fold-Level Scores

To do proper pairwise statistical tests between algorithm families, we need **fold-level scores** -- not just the mean. We run 10-fold cross-validation for nine model families, capturing every fold's R² score. This gives us the paired samples we need for rigorous comparison.

In [ ]:
from sklearn.model_selection import KFold
from ferroml.linear import LinearRegression, RidgeRegression, LassoRegression, ElasticNet
from ferroml.trees import (
    DecisionTreeRegressor, RandomForestRegressor,
    GradientBoostingRegressor, HistGradientBoostingRegressor,
)
from ferroml.neighbors import KNeighborsRegressor

models = {
    "LinearRegression":   lambda: LinearRegression(),
    "Ridge":              lambda: RidgeRegression(alpha=1.0),
    "Lasso":              lambda: LassoRegression(alpha=0.1),
    "ElasticNet":         lambda: ElasticNet(alpha=0.1, l1_ratio=0.5),
    "KNN (k=5)":          lambda: KNeighborsRegressor(n_neighbors=5),
    "DecisionTree":       lambda: DecisionTreeRegressor(max_depth=8),
    "RandomForest":       lambda: RandomForestRegressor(n_estimators=100, max_depth=10),
    "GradientBoosting":   lambda: GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4),
    "HistGBT":            lambda: HistGradientBoostingRegressor(max_iter=100, learning_rate=0.1, max_depth=6),
}

kf = KFold(n_splits=10, shuffle=True, random_state=42)
fold_scores = {name: [] for name in models}

for train_idx, test_idx in kf.split(X):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]
    for name, make_model in models.items():
        m = make_model()
        m.fit(X_tr, y_tr)
        preds = np.array(m.predict(X_te))
        ss_res = np.sum((y_te - preds) ** 2)
        ss_tot = np.sum((y_te - y_te.mean()) ** 2)
        fold_scores[name].append(1.0 - ss_res / ss_tot)

print(f"{'Model':<22} {'Mean R²':>8} {'Std':>8}")
print("-" * 40)
for name in sorted(fold_scores, key=lambda n: -np.mean(fold_scores[n])):
    print(f"{name:<22} {np.mean(fold_scores[name]):>8.4f} {np.std(fold_scores[name]):>8.4f}")

## The Multiple Comparisons Problem

Here's the trap that most model selection ignores.

When you compare 9 models pairwise, you're running **36 hypothesis tests**. At a significance level of 0.05, you'd expect roughly 2 false positives *even if all models were identical*. The more models you try, the worse this gets.

This is why we apply **Benjamini-Hochberg correction** -- it controls the *false discovery rate*, ensuring that among all the differences you declare "significant," the expected proportion of false positives stays below your chosen threshold.

| Correction Method | What it controls | How strict |
|---|---|---|
| **None** | Nothing -- raw p-values | Most liberal (most false positives) |
| **Bonferroni** | Family-wise error rate | Most conservative (misses real differences) |
| **Holm** | Family-wise error rate | Less conservative than Bonferroni |
| **Benjamini-Hochberg** | False discovery rate | Good balance for model selection |

In [ ]:
# Paired t-tests with Benjamini-Hochberg correction
model_names = sorted(fold_scores, key=lambda n: -np.mean(fold_scores[n]))
n_models = len(model_names)
n_comparisons = n_models * (n_models - 1) // 2

# Collect raw p-values from paired t-tests (same folds = paired samples)
raw_pvals = []
pairs = []
for i, j in combinations(range(n_models), 2):
    a = np.array(fold_scores[model_names[i]])
    b = np.array(fold_scores[model_names[j]])
    _, p_val = stats.ttest_rel(a, b)
    raw_pvals.append(p_val)
    pairs.append((i, j))

raw_pvals = np.array(raw_pvals)

# Benjamini-Hochberg correction
def benjamini_hochberg(pvals):
    """Apply BH false discovery rate correction."""
    n = len(pvals)
    order = np.argsort(pvals)
    corrected = np.empty(n)
    cummin = 1.0
    for rank_from_top, idx in enumerate(reversed(order)):
        k = n - rank_from_top
        adj = pvals[idx] * n / k
        cummin = min(cummin, adj)
        corrected[idx] = min(cummin, 1.0)
    return corrected

corrected_pvals = benjamini_hochberg(raw_pvals)

# Build significance and p-value matrices
alpha = 0.05
sig_matrix = np.full((n_models, n_models), False)
pval_matrix = np.ones((n_models, n_models))
for k, (i, j) in enumerate(pairs):
    pval_matrix[i, j] = corrected_pvals[k]
    pval_matrix[j, i] = corrected_pvals[k]
    sig_matrix[i, j] = corrected_pvals[k] < alpha
    sig_matrix[j, i] = corrected_pvals[k] < alpha

# Identify the competitive set
best_name = model_names[0]
competitive_set = {best_name}
for k, (i, j) in enumerate(pairs):
    if i == 0 and not sig_matrix[0, j]:
        competitive_set.add(model_names[j])

# Report
uncorrected_sig = sum(1 for p in raw_pvals if p < 0.05)
corrected_sig = sum(1 for p in corrected_pvals if p < 0.05)

print(f"Total pairwise comparisons: {n_comparisons}")
print(f"Significant (uncorrected): {uncorrected_sig}")
print(f"Significant (BH-corrected): {corrected_sig}")
if uncorrected_sig > corrected_sig:
    print(f"\nWithout correction, you'd claim {uncorrected_sig - corrected_sig} more differences")
    print(f"-- some of which are likely false positives.\n")

print(f"Best model: {best_name} (R² = {np.mean(fold_scores[best_name]):.4f})")
print(f"\nCompetitive set (not significantly worse than best):")
for name in model_names:
    tag = " <-- competitive" if name in competitive_set else ""
    r2 = np.mean(fold_scores[name])
    if name == best_name:
        print(f"  {name:<22} R² = {r2:.4f}  (reference){tag}")
    else:
        p = pval_matrix[0, model_names.index(name)]
        print(f"  {name:<22} R² = {r2:.4f}  p = {p:.4f}{tag}")

print(f"\n{len(competitive_set)} / {n_models} models are competitive with the best.")

## Kaggle Thinking vs. Scientific Thinking

The left panel shows how most leaderboards work: rank by mean score, crown the winner. The right panel shows reality: the top two models have **overlapping confidence intervals** and are not statistically distinguishable.

In [ ]:
means = [np.mean(fold_scores[n]) for n in model_names]
stds = [np.std(fold_scores[n]) for n in model_names]
n_folds = 10
t_crit = stats.t.ppf(0.975, df=n_folds - 1)
ci_half = [t_crit * s / np.sqrt(n_folds) for s in stds]

colors = ["#2ecc71" if n in competitive_set else "#e74c3c" for n in model_names]
y_pos = np.arange(len(model_names))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

# ── Left panel: Kaggle thinking ──
bars1 = ax1.barh(y_pos, means, color="#3498db", alpha=0.85, height=0.65, edgecolor="white")
ax1.set_yticks(y_pos)
ax1.set_yticklabels(model_names, fontsize=11)
ax1.invert_yaxis()
ax1.set_xlabel("R² Score", fontsize=12)
ax1.set_title('Kaggle Thinking\n"Pick the highest number"',
              fontsize=14, fontweight="bold", color="#c0392b")

for i, (m, bar) in enumerate(zip(means, bars1)):
    ax1.text(m + 0.008, i, f"#{i+1}", va="center", fontsize=10, fontweight="bold", color="#2c3e50")

ax1.annotate("WINNER!", xy=(means[0], 0), xytext=(means[0] + 0.06, -0.3),
            fontsize=11, fontweight="bold", color="#c0392b",
            arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.5))

# ── Right panel: Scientific thinking ──
ax2.barh(y_pos, means, xerr=ci_half,
         color=colors, alpha=0.85, capsize=5, edgecolor="white", height=0.65)
ax2.invert_yaxis()
ax2.set_xlabel("R² Score (with 95% CI)", fontsize=12)
ax2.set_title('Scientific Thinking\n"Which models are actually distinguishable?"',
              fontsize=14, fontweight="bold", color="#27ae60")

green_patch = mpatches.Patch(color="#2ecc71", label="Competitive with best (p > 0.05)")
red_patch = mpatches.Patch(color="#e74c3c", label="Significantly worse (BH-corrected)")
ax2.legend(handles=[green_patch, red_patch], loc="lower right", fontsize=10, framealpha=0.9)

plt.suptitle("Two Ways to Read a Model Leaderboard",
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Detailed Uncertainty: Every Fold Tells a Story

Each small dot below is one fold's R² score. The thick line is the 95% confidence interval. The green shaded region is the best model's CI -- any model whose CI overlaps significantly with this region is a candidate for "statistically equivalent."

Notice how individual folds can vary wildly. A model that looks great on average might have terrible folds hidden inside.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for i, name in enumerate(model_names):
    m = means[i]
    lo = m - ci_half[i]
    hi = m + ci_half[i]
    color = "#2ecc71" if name in competitive_set else "#e74c3c"

    # CI bar
    ax.plot([lo, hi], [i, i], color=color, linewidth=3, solid_capstyle="round", alpha=0.7)
    # Mean dot
    ax.scatter([m], [i], color=color, s=100, zorder=5, edgecolors="white", linewidth=1.5)
    # Individual fold scores
    ax.scatter(fold_scores[name], [i] * n_folds,
               color=color, s=15, alpha=0.35, zorder=3)

# Shade the best model's CI
best_lo = means[0] - ci_half[0]
best_hi = means[0] + ci_half[0]
ax.axvspan(best_lo, best_hi, alpha=0.10, color="#2ecc71")

ax.set_yticks(y_pos)
ax.set_yticklabels(model_names, fontsize=11)
ax.invert_yaxis()
ax.set_xlabel("R² Score (10-fold CV)", fontsize=12)
ax.set_title("Model Rankings with Uncertainty\nSmall dots = individual fold scores",
             fontsize=14, fontweight="bold")

ax.legend(handles=[
    mpatches.Patch(color="#2ecc71", alpha=0.3, label="Best model's 95% CI"),
    plt.Line2D([0], [0], color="#2ecc71", lw=3, label="Competitive (p > 0.05)"),
    plt.Line2D([0], [0], color="#e74c3c", lw=3, label="Significantly worse"),
], loc="lower right", fontsize=10, framealpha=0.9)

ax.axvline(x=0, color="gray", linestyle=":", alpha=0.3)
plt.tight_layout()
plt.show()

## Pairwise Significance Heatmap

This heatmap shows the corrected p-value for every model pair. Darker colors = stronger evidence that the models are different. Stars indicate statistical significance after Benjamini-Hochberg correction.

The key insight: the top-left corner (comparisons among the best models) is often *not* significant -- they're too close to tell apart.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

# Use -log10(p) for visual contrast, cap at 4 (p=0.0001)
display = -np.log10(np.clip(pval_matrix, 1e-4, 1.0))
np.fill_diagonal(display, 0)

im = ax.imshow(display, cmap="YlOrRd", aspect="auto", vmin=0, vmax=4)

ax.set_xticks(range(n_models))
ax.set_xticklabels(model_names, rotation=45, ha="right", fontsize=10)
ax.set_yticks(range(n_models))
ax.set_yticklabels(model_names, fontsize=10)

for i in range(n_models):
    for j in range(n_models):
        if i != j:
            p = pval_matrix[i, j]
            if p < 0.001:
                txt = "<.001"
            elif p < 0.01:
                txt = f"{p:.3f}"
            else:
                txt = f"{p:.2f}"
            sig = "**" if p < 0.01 else ("*" if p < 0.05 else "")
            color = "white" if display[i, j] > 2 else "black"
            ax.text(j, i, f"{txt}{sig}", ha="center", va="center",
                    fontsize=8, color=color, fontweight="bold" if sig else "normal")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("-log10(corrected p-value)", fontsize=11)
cbar.ax.axhline(y=-np.log10(0.05), color="black", linestyle="--", linewidth=1.5)
cbar.ax.text(0.5, -np.log10(0.05), " p=0.05", va="bottom", fontsize=9,
             transform=cbar.ax.get_yaxis_transform())

ax.set_title("Pairwise Model Comparisons\n(BH-corrected paired t-tests, * p<0.05, ** p<0.01)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## The Punchline

Let's look at what a naive AutoML would tell you vs. what FerroML tells you:

In [ ]:
best_name = model_names[0]
best_r2 = np.mean(fold_scores[best_name])
second_name = model_names[1]
second_r2 = np.mean(fold_scores[second_name])
p_12 = pval_matrix[0, 1]

print("=" * 65)
print("NAIVE AutoML SAYS:")
print("=" * 65)
print(f"  Best model: {best_name}")
print(f"  Score: {best_r2:.4f}")
print(f"  Ship it!")
print()
print("=" * 65)
print("FerroML SAYS:")
print("=" * 65)
print(f"  Best model:  {best_name:<22} R² = {best_r2:.4f}")
print(f"  Runner-up:   {second_name:<22} R² = {second_r2:.4f}")
print(f"  Difference:  {best_r2 - second_r2:.4f}")
print(f"  Corrected p: {p_12:.4f}")
print()
if p_12 >= 0.05:
    print(f"  These two models are NOT statistically distinguishable.")
    print(f"  Picking {best_name} over {second_name} is coin-flip confidence.")
    print(f"  Consider: which is simpler? faster? more interpretable?")
else:
    print(f"  {best_name} IS significantly better than {second_name}.")
print()
print(f"  Competitive set: {len(competitive_set)} / {n_models} models")
print(f"  Models in the running: {', '.join(n for n in model_names if n in competitive_set)}")
print()
print("Most AutoML picks the highest number.")
print("FerroML picks the best model that's actually distinguishable.")

## Why This Matters

In production ML, the difference between the #1 and #2 model is often noise. But teams spend weeks chasing that last 0.01 of AUC, convinced that the leaderboard leader is meaningfully better.

FerroML's approach gives you:

- **Honest uncertainty** -- every score comes with a confidence interval, not just a point estimate
- **Multiple testing correction** -- when you compare N models, naive p-values inflate false positives. FerroML corrects for this automatically.
- **The competitive set** -- instead of one "best" model, you get the set of models that are statistically indistinguishable from the best. Pick the simplest, fastest, or most interpretable from that set.
- **Built into AutoML** -- this isn't a post-hoc analysis you have to do yourself. `AutoMLConfig(statistical_tests=True)` gives you `result.competitive_models()` out of the box.

**The goal isn't to find the model with the highest number. It's to find the best model you can actually be confident about.**

---

*FerroML: ML with statistical rigor, written in Rust.*